## 1. Basic Tasks

**1. Load the messy e-commerce dataset and identify columns containing nulls and duplicate rows using
.filter(), .distinct(), and .dropDuplicates().**

In [0]:
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/messy_ecommerce_extended.csv",
    header=True,
    inferSchema=True    
)

In [0]:
from pyspark.sql import functions as F

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.display()

In [0]:
from pyspark.sql.functions import *
distinct_df = df.distinct()
null_orders = df.filter(col("order_id").isNotNull())
cleaned_duplicates_orders = null_orders.dropDuplicates(["order_id"])
cleaned_orders = cleaned_duplicates_orders.dropna(how="any")
cleaned_orders.display()

**2. Rename at least 3 columns to a consistent naming convention (e.g., snake_case) using
withColumnRenamed.**

In [0]:
df_renamed = cleaned_orders.withColumnRenamed("customer_id", "customerId").withColumnRenamed("order_id", "orderId").withColumnRenamed("order_date", "orderDate").withColumnRenamed("product_category", "productCategory")
df_renamed.display()

**3. Connect a Databricks Repo to a Git provider and make your first commit of a cleaning notebook.**

For connecting a Databricks repo to GitHub we can do it two ways:
- first is create a repo on Github an then copy the the git repo link and create a git folder in DataBricks and paste the git repo link in DataBricks git folder and in your account and go to Linked account and configure the GitHub and add the add the repo for access. and then create a notbook and perform the tasks and then click on the git link and then put a commit and then click on commit & push.
- In the second way first create a GitHub repo then create a normal folder and in your account and go to Linked account and configure the GitHub and add the add the repo for access. Then click on the terminal and then run the below code part:
```
git init
git add .
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/SurajitM0nd0l/new_repo_name.git
git push -u origin main
```

## 2. Intermediate Tasks

**4. Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove
exact duplicates, and sort by order_date.**

In [0]:
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header = True,
    inferSchema = True
)

In [0]:
from pyspark.sql.functions import col, current_date, coalesce

cleaned_df = (
    df.dropna(how="all")
    .withColumn("order_date", coalesce(col("order_date"), current_date()))
    .fillna({
        "order_id": -1,
        "customer_id": -1,
        "transaction_id": -1,
        "product_id": -1,
        "quantity": 0,
        "discount_amount": 0,
        "total_amount": 0.00
    })
    .dropDuplicates()
    .orderBy("order_date")
)

In [0]:
df.write.mode("overwrite").saveAsTable("dev.silver.cleaned_sales_new")

**5. Perform an aggregation (revenue by category or region) and a join against a second small reference
table (e.g., customers or regions).**

In [0]:
from pyspark.sql.functions import sum

transactions = spark.read.csv(
    "/Volumes/dev/bronze/raw/transaction.csv", 
    header=True, 
    inferSchema=True
)
customers = spark.read.csv(
    "/Volumes/dev/bronze/raw/customers-3.csv", 
    header=True, 
    inferSchema=True
)

transactions_clean = transactions.dropDuplicates().dropna()
customers_clean = customers.dropDuplicates().dropna()

merged_df = transactions_clean.join(customers_clean, "customer_id", "left")

summary_df = merged_df.groupBy("status").agg(sum("amount").alias("total_amount"))

summary_df.write.mode("overwrite").saveAsTable("dev.gold.financial_summary")

**6. Create a feature branch in your Databricks Repo, change the cleaning logic, and open a pull request
describing what changed and why.**